# Held-out architecture comparison — Claude Sonnet 4.5

This presentation notebook compares one tool-using agent with the Research → Math → Verifier pipeline on the frozen Coca-Cola (KO) test set. It also checks the effect of removing the Verifier and enabling extended thinking. Gold answers are used only after each run.

## Experiment rules

KO is the held-out company. Its results must not be used to change prompts, retrieval, tools, formulas, or scoring rules. Each full batch is saved before evaluation. Architecture selection prioritizes numerical precision and correctness; cost and latency are secondary.

## Setup

Load the reusable runtime and evaluation code from `src/`.

In [1]:
import sys
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found.*")
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

In [2]:
from agent.data import load_gold_questions, load_question
from agent.decomposed import DecomposedAgentHarness
from agent.monolithic import MonolithicAgentHarness
from agent.observation import format_observable_timeline, format_run_summary

In [3]:
from evaluation.metrics import (
    evaluate_saved_results, format_evaluation_summary, load_run,
    save_evaluation_results, save_run_records,
)
from pandas import DataFrame

In [4]:
from config import (
    BEDROCK_REASONING_MODE, BEDROCK_SONNET_MODEL_ID,
    DECOMPOSED_MATH_THINKING_BUDGET_TOKENS,
    DECOMPOSED_VERIFIER_THINKING_BUDGET_TOKENS,
    EVALUATION_JUDGE_MODEL_ID,
    EVALUATION_JUDGE_THINKING_BUDGET_TOKENS,
)

In [5]:
comparison_metrics = (
    "run_success_rate", "numerical_correctness_rate",
    "evidence_validity_rate", "evidence_completeness_rate",
    "formula_correctness_rate", "formatting_correctness_rate",
    "audit_trail_completeness_rate", "tool_f1",
    "evidence_recall_at_5", "irrelevant_retrieval_rate_at_5",
    "average_latency_ms", "total_input_tokens", "total_output_tokens",
    "total_reasoning_tokens", "total_estimated_cost_usd",
    "average_estimated_cost_usd",
    "average_judge_llm_answer_correctness_grade",
    "average_judge_workflow_correctness",
    "average_judge_chronological_coherence",
    "average_judge_workflow_efficiency",
    "average_judge_auditability", "judge_total_estimated_cost_usd",
    "judge_average_estimated_cost_usd",
    "judge_total_model_invocations",
)
CHECKS = (
    "numerical_correct", "evidence_valid", "evidence_complete",
    "value_evidence_valid", "periods_valid", "formatting_correct",
    "audit_trail_complete",
)

## Fixed model configuration

Confirm that both primary architectures use Claude Sonnet 4.5 with extended thinking disabled. Haiku 4.5 is used only for the exploratory review.

In [6]:
model_id = BEDROCK_SONNET_MODEL_ID
assert "claude-sonnet-4-5" in model_id
assert BEDROCK_REASONING_MODE == "none"
assert DECOMPOSED_MATH_THINKING_BUDGET_TOKENS == 0
assert DECOMPOSED_VERIFIER_THINKING_BUDGET_TOKENS == 0
assert "claude-haiku-4-5" in EVALUATION_JUDGE_MODEL_ID
assert EVALUATION_JUDGE_THINKING_BUDGET_TOKENS > 0

In [7]:
DataFrame([
    {"model": model_id, "architecture": "monolithic", "thinking": "none"},
    {"model": model_id, "architecture": "decomposed_with_verifier", "thinking": "none"},
])

,model,architecture,thinking
0,us.anthropic.claude-sonnet-4-5-20250929-v1:0,monolithic,none
1,us.anthropic.claude-sonnet-4-5-20250929-v1:0,decomposed_with_verifier,none


## Held-out benchmark

Load the 16 frozen KO questions. Gold records remain separate from the answer pipelines.

In [8]:
ticker = "KO"
gold_records = load_gold_questions(ticker)
gold_by_id = {record["question_id"]: record for record in gold_records}
assert len(gold_records) == 16

In [8]:
DataFrame([
    {"question_id": record["question_id"], "category": record["category"], "question": record["question"]}
    for record in gold_records
])

,question_id,category,question
0,ko-01,direct_extraction,What were The Coca-Cola Company's consolidated...
1,ko-02,direct_extraction,What was The Coca-Cola Company's consolidated ...
2,ko-03,direct_extraction,Which Coca-Cola operating segment generated th...
3,ko-04,direct_extraction,What was Coca-Cola's EMEA operating income in ...
4,ko-05,single_formula,What was The Coca-Cola Company's operating mar...
5,ko-06,single_formula,What was The Coca-Cola Company's net profit ma...
6,ko-07,single_formula,What share of The Coca-Cola Company's fiscal 2...
7,ko-08,single_formula,What was The Coca-Cola Company's operating cas...
8,ko-09,period_comparison,"Between fiscal 2024 and fiscal 2025, how did C..."
9,ko-10,period_comparison,How did Coca-Cola's net income attributable to...


## Reusable harnesses

Create one harness for each architecture. The decomposed pipeline includes the Verifier and allows one Math correction attempt.

In [9]:
monolithic_harness = MonolithicAgentHarness(provider="bedrock", model_id=model_id)
decomposed_harness = DecomposedAgentHarness(
    provider="bedrock", model_id=model_id,
    enable_verifier=True, enable_math_repair=True,
)

In [10]:
preflight = {
    "monolithic": monolithic_harness.preflight(),
    "decomposed": decomposed_harness.preflight(),
}
preflight

{'monolithic': {'ready': True,
  'provider': 'bedrock',
  'model': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0',
  'reasoning_mode': 'none',
  'langfuse': 'authenticated',
  'qdrant_collection': 'sec_filing_chunks_v1'},
 'decomposed': {'ready': True,
  'provider': 'bedrock',
  'model': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0',
  'architecture': 'decomposed',
  'pipeline_variant': 'verification_with_repair',
  'reasoning': {'research': 'none', 'math': 'none', 'verifier': 'none'},
  'langfuse': 'authenticated',
  'qdrant_collection': 'sec_filing_chunks_v1'}}

# Representative question comparison

Run one example from each question type through both architectures. These examples show how each system works and are not included in the final batch scores.

## Direct extraction — KO-01

Ask both systems for one value already present in the frozen facts.

In [11]:
direct_question = load_question("ko-01")
direct_gold = gold_by_id[direct_question.question_id]["answer"]
direct_question.question, direct_gold

("What were The Coca-Cola Company's consolidated net operating revenues in fiscal 2025?",
 {'response_type': 'number',
  'rounding': 'exact USD',
  'unit': 'USD',
  'value': '47941000000'})

In [12]:
monolithic_direct = monolithic_harness.run_question(direct_question.question)
decomposed_direct = decomposed_harness.run_question(direct_question.question)

In [13]:
DataFrame([
    {"architecture": "monolithic", "status": monolithic_direct.status, "answer": monolithic_direct.answer.answer_text if monolithic_direct.answer else None},
    {"architecture": "decomposed", "status": decomposed_direct.status, "answer": decomposed_direct.answer.answer_text if decomposed_direct.answer else None},
])

,architecture,status,answer
0,monolithic,success,The Coca-Cola Company's consolidated net opera...
1,decomposed,success,The Coca-Cola Company's consolidated net opera...


### Monolithic trace

Show the single agent's observable steps.

In [14]:
print(format_observable_timeline(monolithic_direct.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth,
  calculate (new - old) / old * 100, not new / old * 100. Keep every used
  calculator operation, exact inputs, and exact result in calculations. Ensure
  directional wording agrees with results.
- Ground each reported value in returned evidence or a
  calculator result. Cite only returned evidence IDs that directly supp

### Decomposed trace

Show the Research, Math, and Verifier steps.

In [15]:
print(format_observable_timeline(decomposed_direct.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Answer and gold summaries

Compare both answers with the hidden gold record after generation.

In [16]:
print("===== MONOLITHIC =====")
print(format_run_summary(monolithic_direct, direct_gold))
print("===== DECOMPOSED =====")
print(format_run_summary(decomposed_direct, direct_gold))

===== MONOLITHIC =====
Question: What were The Coca-Cola Company's consolidated net operating revenues in fiscal 2025?
Model answer: The Coca-Cola Company's consolidated net operating revenues in fiscal 2025 were $47,941,000,000.
Gold answer: 47941000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-ko-revenue-fy2025
Latency: 7385.14 ms | Tokens: {'inputTokens': 4562, 'outputTokens': 361, 'totalTokens': 4923} | Provider retries: 0
Structured-output repairs: 0 (no schema-driven model reinvocation)
Cost: {'input': '0.013686', 'output': '0.005415', 'total': '0.019101'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/faacd56bc108b3c4481f7fd722b97bd0
===== DECOMPOSED =====
Question: What were The Coca-Cola Company's consolidated net operating revenues in fiscal 2025?
Model answer: The Coca-Cola Company's consolidated net operating revenues in fiscal 2025 were $47,941,000,000.
Gold answer: 47941000000
Gold values: -
Tools: lookup_fact (ok)
Evidence: fact-ko

### Conclusion: direct extraction

Both systems returned the exact FY2025 revenue of \$47,941,000,000 from the same fact. The monolithic run was faster and cheaper: **7.39seconds** and **\$0.0191**, compared with **15.33seconds** and **\$0.0283**. The extra stages added no quality benefit for this simple lookup.

## Single formula — KO-05

Ask both systems to calculate FY2025 operating margin.

In [17]:
formula_question = load_question("ko-05")
formula_gold = gold_by_id[formula_question.question_id]["answer"]
formula_question.question, formula_gold

("What was The Coca-Cola Company's operating margin in fiscal 2025?",
 {'response_type': 'number',
  'rounding': 'nearest 0.01',
  'unit': 'percent',
  'value': '28.71'})

In [18]:
monolithic_formula = monolithic_harness.run_question(formula_question.question)
decomposed_formula = decomposed_harness.run_question(formula_question.question)

In [19]:
DataFrame([
    {"architecture": "monolithic", "status": monolithic_formula.status, "answer": monolithic_formula.answer.answer_text if monolithic_formula.answer else None},
    {"architecture": "decomposed", "status": decomposed_formula.status, "answer": decomposed_formula.answer.answer_text if decomposed_formula.answer else None},
])

,architecture,status,answer
0,monolithic,success,The Coca-Cola Company's operating margin in fi...
1,decomposed,success,The Coca-Cola Company's operating margin in fi...


### Monolithic trace

Inspect the single agent's evidence and calculations.

In [20]:
print(format_observable_timeline(monolithic_formula.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth,
  calculate (new - old) / old * 100, not new / old * 100. Keep every used
  calculator operation, exact inputs, and exact result in calculations. Ensure
  directional wording agrees with results.
- Ground each reported value in returned evidence or a
  calculator result. Cite only returned evidence IDs that directly supp

### Decomposed trace

Inspect the staged evidence, calculation, and verification path.

In [21]:
print(format_observable_timeline(decomposed_formula.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Answer and gold summaries

Compare the final margins and their supporting calculations.

In [22]:
print("===== MONOLITHIC =====")
print(format_run_summary(monolithic_formula, formula_gold))
print("===== DECOMPOSED =====")
print(format_run_summary(decomposed_formula, formula_gold))

===== MONOLITHIC =====
Question: What was The Coca-Cola Company's operating margin in fiscal 2025?
Model answer: The Coca-Cola Company's operating margin in fiscal 2025 was 28.71%. This was calculated by dividing operating income of $13,762,000,000 by revenue of $47,941,000,000.
Gold answer: 28.71
Gold values: -
Tools: lookup_fact (ok), lookup_fact (ok), calculate (ok), calculate (ok)
Evidence: fact-ko-operating-income-fy2025, fact-ko-revenue-fy2025
Latency: 11155.57 ms | Tokens: {'inputTokens': 10826, 'outputTokens': 668, 'totalTokens': 11494} | Provider retries: 0
Structured-output repairs: 0 (no schema-driven model reinvocation)
Cost: {'input': '0.032478', 'output': '0.01002', 'total': '0.042498'}
Trace: https://cloud.langfuse.com/project/cmrva8h6h0hchad0d3guy659p/traces/9f4b05f3fa335b88bfa1ee0feb763a53
===== DECOMPOSED =====
Question: What was The Coca-Cola Company's operating margin in fiscal 2025?
Model answer: The Coca-Cola Company's operating margin in fiscal 2025 was 28.71%. T

### Conclusion: single formula

Both systems returned the exact **28.71% operating margin**. The decomposed run used one direct percentage calculation, while the monolithic run added an unnecessary division. The decomposed run took **16.28 seconds** and cost **\$0.0405**, compared with **11.16 seconds** and **\$0.0425** for the monolithic run.

## Period comparison — KO-12

Compare North America and consolidated operating-income growth across two years.

In [23]:
comparison_question = load_question("ko-12")
comparison_gold = gold_by_id[comparison_question.question_id]["answer"]
comparison_question.question, comparison_gold

("From fiscal 2024 to fiscal 2025, how did North America operating income growth compare with Coca-Cola's consolidated operating income growth?",
 {'required_claims': ['consolidated operating income grew faster than North America operating income',
   'the growth-rate difference was 26.45 percentage points'],
  'required_values': [{'label': 'North America operating income growth',
    'unit': 'percent',
    'value': '11.28'},
   {'label': 'consolidated operating income growth',
    'unit': 'percent',
    'value': '37.73'}],
  'response_type': 'paragraph'})

In [24]:
monolithic_comparison = monolithic_harness.run_question(comparison_question.question)
decomposed_comparison = decomposed_harness.run_question(comparison_question.question)

In [25]:
DataFrame([
    {"architecture": "monolithic", "status": monolithic_comparison.status, "answer": monolithic_comparison.answer.answer_text if monolithic_comparison.answer else None},
    {"architecture": "decomposed", "status": decomposed_comparison.status, "answer": decomposed_comparison.answer.answer_text if decomposed_comparison.answer else None},
])

,architecture,status,answer
0,monolithic,success,"From fiscal 2024 to fiscal 2025, North America..."
1,decomposed,success,"From fiscal 2024 to fiscal 2025, North America..."


### Monolithic trace

Inspect the evidence and calculation sequence.

In [26]:
print(format_observable_timeline(monolithic_comparison.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth,
  calculate (new - old) / old * 100, not new / old * 100. Keep every used
  calculator operation, exact inputs, and exact result in calculations. Ensure
  directional wording agrees with results.
- Ground each reported value in returned evidence or a
  calculator result. Cite only returned evidence IDs that directly supp

### Decomposed trace

Inspect the same work across the three pipeline stages.

In [27]:
print(format_observable_timeline(decomposed_comparison.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Answer and gold summaries

Check both growth rates and the difference between them.

In [28]:
print("===== MONOLITHIC =====")
print(format_run_summary(monolithic_comparison, comparison_gold))
print("===== DECOMPOSED =====")
print(format_run_summary(decomposed_comparison, comparison_gold))

===== MONOLITHIC =====
Question: From fiscal 2024 to fiscal 2025, how did North America operating income growth compare with Coca-Cola's consolidated operating income growth?
Model answer: From fiscal 2024 to fiscal 2025, North America operating income grew 11.28%, while Coca-Cola's consolidated operating income grew 37.73%. North America's operating income increased from $4,556,000,000 in FY2024 to $5,070,000,000 in FY2025, an increase of $514,000,000. Consolidated operating income increased from $9,992,000,000 in FY2024 to $13,762,000,000 in FY2025, an increase of $3,770,000,000. Consolidated operating income growth significantly outpaced North America's growth by 26.45 percentage points. The higher consolidated growth was primarily driven by lower other operating charges due to the prior year remeasurement of the fairlife contingent consideration liability.
Gold answer: consolidated operating income grew faster than North America operating income; the growth-rate difference was 26.4

### Conclusion: period comparison

Both systems returned **11.28% North America growth**, **37.73% consolidated growth**, and the correct **26.45 percentage-point difference**. The decomposed answer was shorter, but its run took **53.85 seconds** and cost **\$0.3258**, versus **37.55 seconds** and **\$0.2132** for the monolithic run. Answer quality was similar; the monolithic path was more efficient.

## Multi-step — KO-16

Compare four growth rates and identify the fastest, slowest, and spread.

In [29]:
multistep_question = load_question("ko-16")
multistep_gold = gold_by_id[multistep_question.question_id]["answer"]
multistep_question.question, multistep_gold

("Did Coca-Cola's net operating revenues, operating income, operating cash flow, and total assets grow at similar rates from fiscal 2024 to fiscal 2025?",
 {'required_claims': ['all four measures increased',
   'operating income grew fastest',
   'net operating revenues grew slowest',
   'the spread between the fastest and slowest growth rates was 35.86 percentage points'],
  'required_values': [{'label': 'net operating revenues growth',
    'unit': 'percent',
    'value': '1.87'},
   {'label': 'operating income growth', 'unit': 'percent', 'value': '37.73'},
   {'label': 'operating cash flow growth', 'unit': 'percent', 'value': '8.86'},
   {'label': 'total assets growth', 'unit': 'percent', 'value': '4.24'}],
  'response_type': 'paragraph'})

In [30]:
monolithic_multistep = monolithic_harness.run_question(multistep_question.question)
decomposed_multistep = decomposed_harness.run_question(multistep_question.question)

In [31]:
DataFrame([
    {"architecture": "monolithic", "status": monolithic_multistep.status, "answer": monolithic_multistep.answer.answer_text if monolithic_multistep.answer else None},
    {"architecture": "decomposed", "status": decomposed_multistep.status, "answer": decomposed_multistep.answer.answer_text if decomposed_multistep.answer else None},
])

,architecture,status,answer
0,monolithic,success,"No, Coca-Cola's net operating revenues, operat..."
1,decomposed,success,"No, Coca-Cola's net operating revenues, operat..."


### Monolithic trace

Inspect the complete single-agent calculation path.

In [32]:
print(format_observable_timeline(monolithic_multistep.observable_timeline))

===== 01 SYSTEM PROMPT =====

You are a read-only SEC filing research agent. Use only the
supplied tools and frozen local data. Do not use outside knowledge, modify data, or
give financial advice, investment recommendations, predictions, or trade guidance.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.
- Use calculate for every arithmetic step. For growth,
  calculate (new - old) / old * 100, not new / old * 100. Keep every used
  calculator operation, exact inputs, and exact result in calculations. Ensure
  directional wording agrees with results.
- Ground each reported value in returned evidence or a
  calculator result. Cite only returned evidence IDs that directly supp

### Decomposed trace

Inspect the complete staged calculation and verification path.

In [33]:
print(format_observable_timeline(decomposed_multistep.observable_timeline))

===== 01 SYSTEM PROMPT [research] attempt 1 =====

You are the research stage of a read-only SEC filing
pipeline. Use only the supplied frozen-data tools. Do not give financial advice,
use outside knowledge, calculate, or answer the user directly.

- Use evidence only for the company in the question. Match its
  requested period and unit; before calculating, confirm every input has the
  requested period. Do not guess when evidence is missing or ambiguous.
- Prefer lookup_fact for normalized facts. Use search_filing when the requested
  concept, granularity, or context is not available through lookup_fact.

Return the exact selected values, units, periods, and only evidence IDs returned by
the tools. Describe the calculation the math stage must perform, but do not perform
it yourself. If evidence is missing or ambiguous, return an empty facts list and
explain that in calculation_request.


===== 02 TOOL DEFINITIONS [research] attempt 1 =====

- search_filing: Search the frozen 10-K for

### Answer and gold summaries

Check the four rates and every required comparison claim.

In [34]:
print("===== MONOLITHIC =====")
print(format_run_summary(monolithic_multistep, multistep_gold))
print("===== DECOMPOSED =====")
print(format_run_summary(decomposed_multistep, multistep_gold))

===== MONOLITHIC =====
Question: Did Coca-Cola's net operating revenues, operating income, operating cash flow, and total assets grow at similar rates from fiscal 2024 to fiscal 2025?
Model answer: No, Coca-Cola's net operating revenues, operating income, operating cash flow, and total assets did not grow at similar rates from fiscal 2024 to fiscal 2025. The growth rates varied significantly:

- Net operating revenues grew 1.87%, from $47,061,000,000 in FY2024 to $47,941,000,000 in FY2025.
- Operating income grew 37.73%, from $9,992,000,000 in FY2024 to $13,762,000,000 in FY2025.
- Operating cash flow grew 8.86%, from $6,805,000,000 in FY2024 to $7,408,000,000 in FY2025.
- Total assets grew 4.24%, from $100,549,000,000 in FY2024 to $104,816,000,000 in FY2025.

Operating income showed the strongest growth at 37.73%, significantly outpacing the other metrics. Operating cash flow grew at 8.86%, total assets at 4.24%, and net operating revenues had the slowest growth at 1.87%.
Gold answer:

### Conclusion: multi-step

Both systems calculated the four growth rates correctly, but both omitted the required **35.86 percentage-point spread**. The monolithic trace contained four failed calculator calls; the decomposed trace had none. The Verifier still approved the incomplete answer and requested no correction. The monolithic run took **35.60 seconds** and cost **\$0.1329**, compared with **52.36 seconds** and **\$0.1418** for the decomposed run. Neither answer fully passed.

# Complete KO held-out experiment

Run all 16 questions once with each primary architecture. These saved batch records—not the examples above—produce the final scores.

## Monolithic batch

Run and save the 16 single-agent answers.

In [35]:
monolithic_runs = monolithic_harness.run_batch(ticker)
assert len(monolithic_runs) == len(gold_records)
monolithic_results_dir = save_run_records(monolithic_runs)

In [36]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status, "failure": run.failure_category}
    for run in monolithic_runs
])

,question_id,status,failure
0,ko-01,success,None
1,ko-02,success,None
2,ko-03,success,None
3,ko-04,success,None
4,ko-05,success,None
5,ko-06,success,None
6,ko-07,success,None
7,ko-08,success,None
8,ko-09,success,None
9,ko-10,success,None


## Decomposed batch

Run and save the same 16 questions with Research, Math, and Verifier stages.

In [37]:
decomposed_runs = decomposed_harness.run_batch(ticker)
assert len(decomposed_runs) == len(gold_records)
decomposed_results_dir = save_run_records(decomposed_runs)

In [38]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status, "failure": run.failure_category}
    for run in decomposed_runs
])

,question_id,status,failure
0,ko-01,success,None
1,ko-02,success,None
2,ko-03,success,None
3,ko-04,success,None
4,ko-05,success,None
5,ko-06,success,None
6,ko-07,success,None
7,ko-08,success,None
8,ko-09,success,None
9,ko-10,success,None


## Inspect every answer

Place both answers beside each question so individual mistakes remain visible before aggregate scoring.

In [39]:
monolithic_answer_rows = [
    {"question_id": run.question.question_id, "architecture": "monolithic",
     "status": run.status, "answer": run.answer.answer_text if run.answer else None}
    for run in monolithic_runs
]

In [40]:
decomposed_answer_rows = [
    {"question_id": run.question.question_id, "architecture": "decomposed",
     "status": run.status, "answer": run.answer.answer_text if run.answer else None}
    for run in decomposed_runs
]

In [41]:
DataFrame(monolithic_answer_rows + decomposed_answer_rows).sort_values(
    ["question_id", "architecture"]
)

,question_id,architecture,status,answer
16,ko-01,decomposed,success,The Coca-Cola Company's consolidated net opera...
0,ko-01,monolithic,success,The Coca-Cola Company's consolidated net opera...
17,ko-02,decomposed,success,The Coca-Cola Company's consolidated operating...
1,ko-02,monolithic,success,The Coca-Cola Company's consolidated operating...
18,ko-03,decomposed,success,North America generated the most operating inc...
2,ko-03,monolithic,success,The North America operating segment generated ...
19,ko-04,decomposed,success,Coca-Cola's EMEA operating income in fiscal 20...
3,ko-04,monolithic,success,Coca-Cola's EMEA operating income in fiscal 20...
20,ko-05,decomposed,success,The Coca-Cola Company's operating margin in fi...
4,ko-05,monolithic,success,The Coca-Cola Company's operating margin in fi...


# Deterministic evaluation

Compare the saved answers, evidence, calculations, tool calls, periods, units, and formatting with the frozen gold records using ordinary Python.

The numerical score is intentionally strict: it rejects missing values and extra structured values, even when an extra value is correct. It therefore measures compliance with the expected answer shape as well as financial accuracy.

In [5]:
monolithic_evaluation = evaluate_saved_results(
    results_directory=monolithic_results_dir
)
monolithic_summary = monolithic_evaluation["summary"]
monolithic_question_metrics = monolithic_evaluation["question_rows"]

In [6]:
decomposed_evaluation = evaluate_saved_results(
    results_directory=decomposed_results_dir
)
decomposed_summary = decomposed_evaluation["summary"]
decomposed_question_metrics = decomposed_evaluation["question_rows"]

## Exploratory Haiku 4.5 workflow review

Use Haiku to score answer quality and workflow clarity from 1 to 5. These scores help explain the traces but cannot override deterministic failures.

In [68]:
monolithic_judged = evaluate_saved_results(
    results_directory=monolithic_results_dir, rerun_judge=True,
)
monolithic_judge_metrics = monolithic_judged["judge_rows"] or []
monolithic_summary = monolithic_judged["summary"]

In [69]:
decomposed_judged = evaluate_saved_results(
    results_directory=decomposed_results_dir, rerun_judge=True,
)
decomposed_judge_metrics = decomposed_judged["judge_rows"] or []
decomposed_summary = decomposed_judged["summary"]

In [8]:
primary_judge_rows = [
    *({"architecture": "monolithic", **row} for row in monolithic_judge_metrics),
    *({"architecture": "decomposed", **row} for row in decomposed_judge_metrics),
]
DataFrame(primary_judge_rows, columns=(
    "architecture", "question_id", "llm_answer_correctness_grade",
    "workflow_correctness", "chronological_coherence",
    "workflow_efficiency", "auditability", "retries", "failure", "explanation",
))

,architecture,question_id,llm_answer_correctness_grade,workflow_correctness,chronological_coherence,workflow_efficiency,auditability,retries,failure,explanation
0,monolithic,ko-01,5,5,5,5,5,0,None,The system correctly executed a straightforwar...
1,monolithic,ko-02,5,5,5,5,5,0,None,The run correctly answered the direct extracti...
2,monolithic,ko-03,5,4,5,4,3,0,None,The answer correctly identifies North America ...
3,monolithic,ko-04,5,5,5,5,5,0,None,The answer precisely matches the gold requirem...
4,monolithic,ko-05,5,5,5,4,5,0,None,The run produces the exact gold answer of 28.7...
5,monolithic,ko-06,5,5,5,5,5,0,None,The run correctly answers the question with 27...
6,monolithic,ko-07,3,3,5,5,5,0,None,The workflow correctly retrieved FY2025 revenu...
7,monolithic,ko-08,4,4,5,5,2,0,None,The final numeric answer of 56.52% is correct ...
8,monolithic,ko-09,5,5,5,4,4,0,None,The generated answer is mathematically correct...
9,monolithic,ko-10,4,4,5,5,5,0,None,The answer correctly identifies that net incom...


In [9]:
save_evaluation_results(
    monolithic_results_dir, monolithic_question_metrics, monolithic_summary,
    monolithic_evaluation["retrieval_rows"], judge_rows=monolithic_judge_metrics,
)
save_evaluation_results(
    decomposed_results_dir, decomposed_question_metrics, decomposed_summary,
    decomposed_evaluation["retrieval_rows"], judge_rows=decomposed_judge_metrics,
)

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-09-01T020801802049Z-decomposed-verification_with_repair-us-anthropic-claude-sonnet-4-5-20250929-v1-0-KO')

## Final metrics

Higher is better except for unsupported-number rate, irrelevant-retrieval rate, latency, tokens, and cost. Judge cost is evaluation overhead and is separate from answer-generation cost.

In [10]:
print("===== MONOLITHIC =====")
print(format_evaluation_summary(monolithic_summary))
print("===== DECOMPOSED =====")
print(format_evaluation_summary(decomposed_summary))

===== MONOLITHIC =====
Runs: 16
run_success_rate: 100.00%
numerical_correctness_rate: 37.50%
evidence_validity_rate: 100.00%
evidence_completeness_rate: 100.00%
formula_correctness_rate: 58.33%
formatting_correctness_rate: 87.50%
unsupported_number_rate: 6.90%
audit_trail_completeness_rate: 56.25%
math_repair_invocation_rate: 0.00%
math_repair_success_rate: 0.00%
tool_precision: 96.77%
tool_recall: 100.00%
tool_f1: 98.36%
tool_exact_match_rate: 93.75%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 100.00%
conditional_execution_rate: 97.92%
evidence_recall_at_5: 75.00%
irrelevant_retrieval_rate_at_5: 90.00%
judge_success_rate: 100.00%
judge_cost_coverage_rate: 100.00%
structured_output_repair_rate: 6.25%
Average latency: 17178.55 ms
Input tokens: 261571
Output tokens: 19283
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 1
Infrastructure failures: 0
Estimated cost: $1.073958
Judge llm answer correctness grade: 4.25/5
Judge workflow correctness: 4.19/5
Judge chronol

## Main metric comparison

Compare the two primary architectures on the same saved KO batch.

In [12]:
DataFrame([
    {"metric": metric, "monolithic": monolithic_summary.get(metric),
     "decomposed": decomposed_summary.get(metric)}
    for metric in comparison_metrics
]).set_index("metric")

,monolithic,decomposed
metric,,
run_success_rate,1.0,1.0
numerical_correctness_rate,0.375,0.6875
evidence_validity_rate,1.0,1.0
evidence_completeness_rate,1.0,1.0
formula_correctness_rate,0.583333,0.666667
formatting_correctness_rate,0.875,0.875
audit_trail_completeness_rate,0.5625,0.6875
tool_f1,0.983607,0.983607
evidence_recall_at_5,0.75,1.0


## Metrics by question type

Show whether failures are concentrated in lookup, one-formula, comparison, or multi-step questions.

In [13]:
category_rows = [
    {"architecture": architecture, "question_type": category, **metrics}
    for architecture, summary in (("monolithic", monolithic_summary), ("decomposed", decomposed_summary))
    for category, metrics in summary["by_category"].items()
]
DataFrame(category_rows).set_index(["question_type", "architecture"])

,,run_count,numerical_correctness_rate,evidence_completeness_rate,formula_correctness_rate,formatting_correctness_rate
question_type,architecture,,,,,
direct_extraction,monolithic,4,1.00,1.0,NaN,1.0
multi_step,monolithic,4,0.00,1.0,0.25,0.5
period_comparison,monolithic,4,0.00,1.0,0.50,1.0
single_formula,monolithic,4,0.50,1.0,1.00,1.0
direct_extraction,decomposed,4,1.00,1.0,NaN,1.0
multi_step,decomposed,4,0.25,1.0,0.25,0.5
period_comparison,decomposed,4,0.50,1.0,0.75,1.0
single_formula,decomposed,4,1.00,1.0,1.00,1.0


## Questions needing review

List every run with a failed answer, evidence, period, formatting, or audit check. Questions not listed passed all selected checks.

In [15]:
all_metric_rows = [
    *({"architecture": "monolithic", **row} for row in monolithic_question_metrics),
    *({"architecture": "decomposed", **row} for row in decomposed_question_metrics),
]
review_rows = [row for row in all_metric_rows if row["status"] != "success" or any(row[name] is not True for name in CHECKS)]

In [16]:
DataFrame(review_rows, columns=(
    "architecture", "question_id", "category", "status",
    "numerical_correct", "evidence_complete", "formula_correct",
    "formatting_correct", "audit_trail_complete",
))

,architecture,question_id,category,status,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete
0,monolithic,ko-03,direct_extraction,success,True,True,None,True,False
1,monolithic,ko-07,single_formula,success,False,True,True,True,True
2,monolithic,ko-08,single_formula,success,False,True,True,True,False
3,monolithic,ko-09,period_comparison,success,False,True,False,True,False
4,monolithic,ko-10,period_comparison,success,False,True,True,True,True
5,monolithic,ko-11,period_comparison,success,False,True,True,True,True
6,monolithic,ko-12,period_comparison,success,False,True,False,True,False
7,monolithic,ko-13,multi_step,success,False,True,True,True,True
8,monolithic,ko-14,multi_step,success,False,True,False,False,False
9,monolithic,ko-15,multi_step,success,False,True,False,False,False


## Failures

Show pipeline, provider, and judge failures separately. An empty table means every call completed, not that every answer was correct.

In [17]:
failure_rows = [
    {"architecture": architecture, "source": "pipeline",
     "question_id": run.question.question_id,
     "failure_category": run.failure_category, "failure_message": run.failure_message}
    for architecture, runs in (("monolithic", monolithic_runs), ("decomposed", decomposed_runs))
    for run in runs if run.status == "failure"
] + [
    {"architecture": architecture, "source": "judge",
     "question_id": row["question_id"], "failure_category": "judge_failure",
     "failure_message": row["failure"]}
    for architecture, rows in (("monolithic", monolithic_judge_metrics),
                               ("decomposed", decomposed_judge_metrics))
    for row in rows if row["failure"] is not None
]
DataFrame(failure_rows, columns=("architecture", "source", "question_id",
                                  "failure_category", "failure_message"))

,architecture,source,question_id,failure_category,failure_message


## Saved experiment artifacts

Display the directories used for the primary evaluation.

In [18]:
DataFrame([
    {"architecture": "monolithic", "results_directory": str(monolithic_results_dir)},
    {"architecture": "decomposed", "results_directory": str(decomposed_results_dir)},
])

,architecture,results_directory
0,monolithic,/home/atanas-vitanov/Documents/final_project_f...
1,decomposed,/home/atanas-vitanov/Documents/final_project_f...


## Primary comparison findings and conclusion

All 32 runs completed and cited valid evidence. The decomposed pipeline achieved higher strict numerical correctness (**68.75% versus 37.50%**), formula correctness (**66.67% versus 58.33%**), and audit-trail completeness (**68.75% versus 56.25%**). It performed best on single-formula and period-comparison questions; both systems remained weak on multi-step work.

This improvement cost time and money. The decomposed pipeline averaged **29.14 seconds** and **\$0.0991** per answer, compared with **17.18 seconds** and **\$0.0671** for the monolithic agent. Both retrieval paths were noisy, with irrelevant-retrieval rates above 88%.

The strict numerical score needs context. Many failures were correct answers with extra supporting values, while KO-03 used a valid filing table that the gold record does not list as an accepted alternative. KO-10 and KO-13 also expose a gold-schema inconsistency: their required claims include comparison values that can be marked “unexpected” when stored as structured outputs.

### Monolithic questions needing review

- **KO-03:** the answer is correct; the audit failure comes from the missing alternative-source mapping.
- **KO-07:** 15.45% is correct, but raw inputs were added as extra answer values.
- **KO-08:** 56.52% is correct, but extra inputs and source mapping reduced the strict score.
- **KO-09:** both rates are correct, but the 35.86-point difference lacks a matching final calculation.
- **KO-10:** both rates are correct, but extra raw inputs fail the expected answer shape.
- **KO-11:** the change and 4.24% are correct, but the two annual asset values are extra outputs.
- **KO-12:** both rates and the prose comparison are correct, but the final difference was not calculated.
- **KO-13:** all four margins are correct; two margin changes are treated as extra structured values.
- **KO-14:** 45.6%, 36.8%, and −8.8 points are too coarsely rounded; gold requires 45.60%, 36.84%, and −8.76.
- **KO-15:** the prose is correct, but one structured field uses percent where the required value is a dollar difference.
- **KO-16:** all four rates are correct, but the 35.86-point spread and its calculation are missing.

### Verified decomposed questions needing review

- **KO-03:** the answer is correct; the same alternative-source mapping causes the audit failure.
- **KO-10:** the answer is correct, but its 14.43-point comparison is treated as an extra structured value.
- **KO-11:** the required change and rate are correct; the two annual asset values are extra outputs.
- **KO-12:** the prose is correct, but the final calculator subtraction uses the reverse sign.
- **KO-13:** the content is correct, but extra change values and a non-canonical calculation path fail strict checks.
- **KO-14:** the same rounding and missing-percentage-calculation error passed verification.
- **KO-15:** all required values are present; an extra FY2025 value and negative-value formatting fail strict output checks.
- **KO-16:** the four rates are correct, but the required spread is missing and still passed verification.

The verified architecture produced the stronger primary numerical and audit result, so it is preferred to the monolithic baseline. The Verifier requested no Math repairs and approved several incomplete or inconsistent answers, so this result does not yet prove that the Verifier itself is an effective correction gate.

# Verifier ablation — decomposed without verification

Run the same 16 questions without the Verifier or Math reinvocation. This optional check isolates the Verifier's observed contribution.

## No-verifier harness

Keep the same model and tools while disabling verification and repair.

In [9]:
no_verifier_harness = DecomposedAgentHarness(
    provider="bedrock", model_id=model_id,
    enable_verifier=False, enable_math_repair=False,
)
no_verifier_harness.preflight()

{'ready': True,
 'provider': 'bedrock',
 'model': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0',
 'architecture': 'decomposed',
 'pipeline_variant': 'no_verification',
 'reasoning': {'research': 'none', 'math': 'none', 'verifier': 'disabled'},
 'langfuse': 'authenticated',
 'qdrant_collection': 'sec_filing_chunks_v1'}

In [10]:
DataFrame([{
    "model": no_verifier_harness.model_id,
    "pipeline_variant": no_verifier_harness.pipeline_variant,
    "math_thinking": no_verifier_harness.math_reasoning_mode,
    "verifier": no_verifier_harness.verifier_reasoning_mode,
}])

,model,pipeline_variant,math_thinking,verifier
0,us.anthropic.claude-sonnet-4-5-20250929-v1:0,no_verification,none,disabled


## Complete no-verifier batch

Run and save all 16 answers, then reload the latest matching saved batch for evaluation.

In [11]:
no_verifier_runs = no_verifier_harness.run_batch(ticker)
assert len(no_verifier_runs) == len(gold_records)
no_verifier_results_dir = save_run_records(no_verifier_runs)

In [8]:
no_verifier_saved = load_run(
    latest=True, architecture="decomposed", ticker=ticker,
    pipeline_variant="no_verification",
)
no_verifier_results_dir = no_verifier_saved["results_directory"]
no_verifier_runs = no_verifier_saved["records"]
no_verifier_saved_summary = no_verifier_saved["summary"]
no_verifier_saved_judge_metrics = no_verifier_saved["judge_rows"]
DataFrame([
    {"question_id": run.question.question_id, "status": run.status,
     "failure": run.failure_category}
    for run in no_verifier_runs
])

,question_id,status,failure
0,ko-01,success,None
1,ko-02,success,None
2,ko-03,success,None
3,ko-04,success,None
4,ko-05,success,None
5,ko-06,success,None
6,ko-07,success,None
7,ko-08,success,None
8,ko-09,success,None
9,ko-10,success,None


## Inspect every no-verifier answer

Review the full answer text before looking at aggregate scores.

In [9]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status,
     "answer": run.answer.answer_text if run.answer else None}
    for run in no_verifier_runs
])

,question_id,status,answer
0,ko-01,success,The Coca-Cola Company's consolidated net opera...
1,ko-02,success,The Coca-Cola Company's consolidated operating...
2,ko-03,success,North America generated the most operating inc...
3,ko-04,success,Coca-Cola's EMEA operating income in fiscal 20...
4,ko-05,success,The Coca-Cola Company's operating margin in fi...
5,ko-06,success,The Coca-Cola Company's net profit margin in f...
6,ko-07,success,The Coca-Cola Company's fiscal 2025 operating ...
7,ko-08,success,The Coca-Cola Company's operating cash flow as...
8,ko-09,success,"Between fiscal 2024 and fiscal 2025, Coca-Cola..."
9,ko-10,success,Coca-Cola's net income attributable to shareow...


## Deterministic no-verifier evaluation

Apply the same frozen checks used for the primary comparison.

In [10]:
no_verifier_evaluation = evaluate_saved_results(
    results_directory=no_verifier_results_dir
)
no_verifier_summary = no_verifier_evaluation["summary"]
no_verifier_question_metrics = no_verifier_evaluation["question_rows"]
no_verifier_evaluation_df = DataFrame(no_verifier_question_metrics)
no_verifier_evaluation_df

,run_id,question_id,category,architecture,pipeline_variant,status,answer_text,numerical_correct,numerical_issues,unexpected_values,...,tool_f1,tool_exact_match,latency_ms,input_tokens,output_tokens,reasoning_tokens,retries,structured_output_repairs,cost_usd,failure_category
0,ko-01-20260901T130452727011Z,ko-01,direct_extraction,decomposed,no_verification,success,The Coca-Cola Company's consolidated net opera...,True,[],[],...,1.0,True,12646.53,4907,573,0,0,0,0.023316,None
1,ko-02-20260901T130505374549Z,ko-02,direct_extraction,decomposed,no_verification,success,The Coca-Cola Company's consolidated operating...,True,[],[],...,1.0,True,10464.69,4919,579,0,0,0,0.023442,None
2,ko-03-20260901T130515840033Z,ko-03,direct_extraction,decomposed,no_verification,success,North America generated the most operating inc...,True,[],[],...,1.0,True,29228.90,47753,1638,0,0,0,0.167829,None
3,ko-04-20260901T130545074072Z,ko-04,direct_extraction,decomposed,no_verification,success,Coca-Cola's EMEA operating income in fiscal 20...,True,[],[],...,1.0,True,13361.12,14545,693,0,0,0,0.054030,None
4,ko-05-20260901T130558437084Z,ko-05,single_formula,decomposed,no_verification,success,The Coca-Cola Company's operating margin in fi...,True,[],[],...,1.0,True,22038.04,14576,1178,0,0,0,0.061398,None
5,ko-06-20260901T130620476470Z,ko-06,single_formula,decomposed,no_verification,success,The Coca-Cola Company's net profit margin in f...,True,[],[],...,1.0,True,27116.64,17357,1325,0,0,0,0.071946,None
6,ko-07-20260901T130647594266Z,ko-07,single_formula,decomposed,no_verification,success,The Coca-Cola Company's fiscal 2025 operating ...,True,[],[],...,1.0,True,18999.30,12223,1080,0,0,0,0.052869,None
7,ko-08-20260901T130706594658Z,ko-08,single_formula,decomposed,no_verification,success,The Coca-Cola Company's operating cash flow as...,True,[],[],...,0.8,False,26768.41,17555,1367,0,0,0,0.073170,None
8,ko-09-20260901T130733364961Z,ko-09,period_comparison,decomposed,no_verification,success,"Between fiscal 2024 and fiscal 2025, Coca-Cola...",False,[],[Difference in Growth Rates=35.86 percent],...,1.0,True,33529.30,20311,2364,0,0,0,0.096393,None
9,ko-10-20260901T130806895769Z,ko-10,period_comparison,decomposed,no_verification,success,Coca-Cola's net income attributable to shareow...,True,[],[],...,0.8,False,44301.24,33721,2806,0,0,0,0.143253,None


### Evaluation by question

Show the result and any extra structured values for every question.

In [11]:
no_verifier_evaluation_df[[
    "question_id", "category", "status",
    "numerical_correct", "evidence_complete",
    "formula_correct", "formatting_correct",
    "audit_trail_complete", "unexpected_values",
]]

,question_id,category,status,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete,unexpected_values
0,ko-01,direct_extraction,success,True,True,None,True,True,[]
1,ko-02,direct_extraction,success,True,True,None,True,True,[]
2,ko-03,direct_extraction,success,True,True,None,True,False,[]
3,ko-04,direct_extraction,success,True,True,None,True,True,[]
4,ko-05,single_formula,success,True,True,True,True,True,[]
5,ko-06,single_formula,success,True,True,True,True,True,[]
6,ko-07,single_formula,success,True,True,True,True,True,[]
7,ko-08,single_formula,success,True,True,True,True,True,[]
8,ko-09,period_comparison,success,False,True,True,True,True,[Difference in Growth Rates=35.86 percent]
9,ko-10,period_comparison,success,True,True,True,True,False,[]


## Haiku 4.5 no-verifier review

Apply the same exploratory judge model and rubric.

In [12]:
if no_verifier_saved_judge_metrics:
    no_verifier_judge_metrics = no_verifier_saved_judge_metrics
    no_verifier_summary = no_verifier_saved_summary
else:
    no_verifier_judged = evaluate_saved_results(
        results_directory=no_verifier_results_dir, rerun_judge=True,
    )
    no_verifier_judge_metrics = no_verifier_judged["judge_rows"] or []
    no_verifier_summary = no_verifier_judged["summary"]

In [13]:
DataFrame(no_verifier_judge_metrics, columns=(
    "question_id", "llm_answer_correctness_grade",
    "workflow_correctness", "chronological_coherence",
    "workflow_efficiency", "auditability", "retries", "failure", "explanation",
))

,question_id,llm_answer_correctness_grade,workflow_correctness,chronological_coherence,workflow_efficiency,auditability,retries,failure,explanation
0,ko-01,5,5,5,5,5,0,None,"The answer of $47,941,000,000 matches the gold..."
1,ko-02,5,5,5,5,5,0,None,"The answer of $13,762,000,000 is numerically c..."
2,ko-03,5,4,5,4,3,0,None,The answer is semantically and numerically cor...
3,ko-04,5,5,5,5,5,0,None,Direct extraction task executed flawlessly: se...
4,ko-05,5,5,4,3,4,0,None,The final answer of 28.71% matches the gold va...
5,ko-06,5,4,4,2,5,0,None,Final answer of 27.34% is correct and properly...
6,ko-07,5,5,5,4,5,0,None,The answer is fully correct: 15.45% matches th...
7,ko-08,5,5,5,4,5,0,None,The answer of 56.52% is numerically correct an...
8,ko-09,5,5,5,5,5,0,None,The generated answer correctly addresses all g...
9,ko-10,5,4,5,3,4,0,None,"All numerical values (23.29%, 8.86%, 14.43pp d..."


In [14]:
save_evaluation_results(
    no_verifier_results_dir, no_verifier_question_metrics,
    no_verifier_summary, no_verifier_evaluation["retrieval_rows"],
    judge_rows=no_verifier_judge_metrics,
)

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-09-01T131243821640Z-decomposed-no_verification-us-anthropic-claude-sonnet-4-5-20250929-v1-0-KO')

## Final metrics

Show the complete saved result for the no-verifier batch.

In [15]:
saved_no_verifier_evaluation = evaluate_saved_results(
    results_directory=no_verifier_results_dir
)
no_verifier_question_metrics = saved_no_verifier_evaluation["question_rows"]
no_verifier_summary = (
    no_verifier_judged["summary"]
    if "no_verifier_judged" in globals()
    else no_verifier_saved_summary
    if no_verifier_saved_judge_metrics
    else saved_no_verifier_evaluation["summary"]
)
no_verifier_evaluation_df = DataFrame(no_verifier_question_metrics)
print(format_evaluation_summary(no_verifier_summary))

Runs: 16
run_success_rate: 100.00%
numerical_correctness_rate: 68.75%
evidence_validity_rate: 100.00%
evidence_completeness_rate: 100.00%
formula_correctness_rate: 75.00%
formatting_correctness_rate: 87.50%
unsupported_number_rate: 8.11%
audit_trail_completeness_rate: 68.75%
math_repair_invocation_rate: 0.00%
math_repair_success_rate: 0.00%
tool_precision: 93.75%
tool_recall: 100.00%
tool_f1: 96.77%
tool_exact_match_rate: 87.50%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 98.52%
conditional_execution_rate: 95.83%
evidence_recall_at_5: 100.00%
irrelevant_retrieval_rate_at_5: 88.67%
judge_success_rate: 100.00%
judge_cost_coverage_rate: 100.00%
structured_output_repair_rate: 0.00%
Average latency: 29441.10 ms
Input tokens: 391559
Output tokens: 29843
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 0
Infrastructure failures: 0
Estimated cost: $1.622322
Judge llm answer correctness grade: 4.81/5
Judge workflow correctness: 4.44/5
Judge chronological coherence: 4.81/

## Verified versus no-verifier comparison

Compare the saved decomposed batches without rerunning either answer pipeline.

In [18]:
if "decomposed_summary" not in globals():
    saved_verified_evaluation = load_run(
        latest=True, architecture="decomposed", ticker=ticker,
        pipeline_variant="verification_with_repair",
        reasoning_mode="research=none, math=none, verifier=none",
    )
    decomposed_summary = saved_verified_evaluation["summary"]
ablation_metrics = (*comparison_metrics,
    "math_repair_invocation_rate", "math_repair_success_rate",
    "structured_output_repair_rate", "total_retries",
)

In [19]:
DataFrame([
    {"metric": metric, "with_verifier": decomposed_summary.get(metric),
     "without_verifier": no_verifier_summary.get(metric)}
    for metric in ablation_metrics
]).set_index("metric")

,with_verifier,without_verifier
metric,,
run_success_rate,1.0,1.0
numerical_correctness_rate,0.6875,0.6875
evidence_validity_rate,1.0,1.0
evidence_completeness_rate,1.0,1.0
formula_correctness_rate,0.666667,0.75
formatting_correctness_rate,0.875,0.875
audit_trail_completeness_rate,0.6875,0.6875
tool_f1,0.983607,0.967742
evidence_recall_at_5,1.0,1.0


### Metrics by question type

Show where removing the Verifier changed the results.

In [20]:
ablation_category_rows = [
    {"variant": variant, "question_type": category, **metrics}
    for variant, summary in (("with_verifier", decomposed_summary), ("without_verifier", no_verifier_summary))
    for category, metrics in summary["by_category"].items()
]
DataFrame(ablation_category_rows).set_index(["question_type", "variant"])

,,run_count,numerical_correctness_rate,evidence_completeness_rate,formula_correctness_rate,formatting_correctness_rate
question_type,variant,,,,,
direct_extraction,with_verifier,4,1.00,1.0,NaN,1.0
multi_step,with_verifier,4,0.25,1.0,0.25,0.5
period_comparison,with_verifier,4,0.50,1.0,0.75,1.0
single_formula,with_verifier,4,1.00,1.0,1.00,1.0
direct_extraction,without_verifier,4,1.00,1.0,NaN,1.0
multi_step,without_verifier,4,0.25,1.0,0.25,0.5
period_comparison,without_verifier,4,0.50,1.0,1.00,1.0
single_formula,without_verifier,4,1.00,1.0,1.00,1.0


## No-verifier questions needing review

Questions not listed passed all selected checks.

In [23]:
no_verifier_review = [
    row for row in no_verifier_question_metrics
    if row["status"] != "success"
    or any(row[name] is not True for name in CHECKS)
]
DataFrame(no_verifier_review, columns=(
    "question_id", "category", "status", "numerical_correct",
    "evidence_complete", "formula_correct", "formatting_correct",
    "audit_trail_complete",
))

,question_id,category,status,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete
0,ko-03,direct_extraction,success,True,True,None,True,False
1,ko-09,period_comparison,success,False,True,True,True,True
2,ko-10,period_comparison,success,True,True,True,True,False
3,ko-11,period_comparison,success,False,True,True,True,True
4,ko-13,multi_step,success,False,True,True,True,True
5,ko-14,multi_step,success,False,True,False,False,False
6,ko-15,multi_step,success,False,True,False,False,False
7,ko-16,multi_step,success,True,True,False,True,False


## Failures

An empty table means all pipeline and judge calls completed.

In [24]:
no_verifier_failures = [
    {"source": "pipeline", "question_id": run.question.question_id,
     "failure_category": run.failure_category, "failure_message": run.failure_message}
    for run in no_verifier_runs if run.status == "failure"
] + [
    {"source": "judge", "question_id": row["question_id"],
     "failure_category": "judge_failure", "failure_message": row["failure"]}
    for row in no_verifier_judge_metrics if row["failure"] is not None
]
no_verifier_failures_df = DataFrame(no_verifier_failures, columns=("source", "question_id",
    "failure_category", "failure_message"))
no_verifier_failures_df

,source,question_id,failure_category,failure_message


## Saved ablation artifacts

Save the evaluation and failure tables beside the run records.

In [25]:
no_verifier_evaluation_df.to_csv(no_verifier_results_dir / "evaluations.csv", index=False)
no_verifier_failures_df.to_csv(no_verifier_results_dir / "failures.csv", index=False)
DataFrame([{
    "artifact": name,
    "path": str(no_verifier_results_dir / name),
} for name in ("runs.jsonl", "summary.json", "judge_records.jsonl",
              "evaluations.csv", "failures.csv")])

,artifact,path
0,runs.jsonl,/home/atanas-vitanov/Documents/final_project_f...
1,summary.json,/home/atanas-vitanov/Documents/final_project_f...
2,judge_records.jsonl,/home/atanas-vitanov/Documents/final_project_f...
3,evaluations.csv,/home/atanas-vitanov/Documents/final_project_f...
4,failures.csv,/home/atanas-vitanov/Documents/final_project_f...


# Verifier ablation findings and conclusion

Removing the Verifier left strict numerical correctness and audit completeness unchanged at **68.75%**. The no-verifier pipeline had higher formula correctness (**75.00% versus 66.67%**), while both variants had complete evidence. The Verifier requested no Math repair in any run, so this batch shows no correction benefit.

The verified run was slightly faster and cheaper: **29.14** versus **29.44 seconds** and **\$0.0991** versus **\$0.1014** per answer. These small reversals can occur because model generations vary; removing one stage does not guarantee a shorter response from the remaining stages. Haiku preferred the no-verifier answers (4.81/5 versus 4.56/5) and auditability (4.50/5 versus 4.38/5), while the verified workflow score was slightly higher (4.50/5 versus 4.44/5).

Every no-verifier review row explained:

- **KO-03:** the answer is correct; the benchmark does not recognize its valid alternative filing table.
- **KO-09:** both rates and the 35.86-point difference are correct; that difference is marked as an extra structured value.
- **KO-10:** the answer and calculations are correct, but one filing source does not expose both periods in its metadata.
- **KO-11:** the change and rate are correct; the two annual asset values are extra outputs.
- **KO-13:** all requested margins and changes are correct; the two required changes are nevertheless treated as extra structured values.
- **KO-14:** 45.6%, 36.8%, and −8.8 points are too coarsely rounded, and the percentage calculations are missing.
- **KO-15:** the required values are correct, but the answer adds FY2025 cash flow and 86.10%, then uses a non-canonical final formula.
- **KO-16:** the four rates are correct, but the 35.86-point spread and its calculation are missing.

### Three-architecture comparison

| Metric | Monolithic | Decomposed + Verifier | Decomposed without Verifier |
|---|---:|---:|---:|
| Successful runs | 16/16 | 16/16 | 16/16 |
| Numerical correctness | 37.50% | **68.75%** | **68.75%** |
| Formula correctness | 58.33% | 66.67% | **75.00%** |
| Audit-trail completeness | 56.25% | **68.75%** | **68.75%** |
| Evidence completeness | 100.00% | 100.00% | 100.00% |
| Tool F1 | **98.36%** | **98.36%** | 96.77% |
| Evidence Recall@5 | 75.00% | **100.00%** | **100.00%** |
| Irrelevant Retrieval Rate@5 | 90.00% | **88.33%** | 88.67% |
| Average answer latency | **17.18 s** | 29.14 s | 29.44 s |
| Total answer tokens | **280,854** | 399,640 | 421,402 |
| Total answer cost | **\$1.0740** | \$1.5859 | \$1.6223 |
| Average answer cost | **\$0.0671** | \$0.0991 | \$0.1014 |
| Total Haiku judge cost | \$0.2326 | \$0.2372 | **\$0.2313** |
| Average answer + judge cost | **\$0.0817** | \$0.1139 | \$0.1159 |
| Haiku answer grade | 4.25/5 | 4.56/5 | **4.81/5** |
| Haiku workflow grade | 4.19/5 | **4.50/5** | 4.44/5 |
| Haiku auditability grade | 4.38/5 | 4.38/5 | **4.50/5** |

Formula correctness covers only the 12 calculation questions. Haiku grades help interpret the traces and do not replace deterministic checks.

### Decision

With numerical precision and correctness first, the no-verifier pipeline is the stronger decomposed variant in this ablation. It tied the verified pipeline on strict numerical correctness, scored higher on formula correctness, and received the higher Haiku answer grade. Its small cost and latency disadvantage is secondary. The verified pipeline still beats the monolithic baseline, but this batch does not show that the Verifier caused that improvement.

# Reasoning ablation — does extended thinking help?

Run the verified decomposed pipeline again with 4,096 thinking tokens for Math and 6,000 for the Verifier. Research, data, tools, prompts, repair policy, and scoring remain unchanged. Gold answers stay outside the answer pipeline.

## Reasoning-enabled harness

Enable extended thinking only for Math and the Verifier.

In [9]:
reasoning_math_budget = 4096
reasoning_verifier_budget = 6000

In [10]:
reasoning_harness = DecomposedAgentHarness(
    provider="bedrock", model_id=model_id, enable_verifier=True, enable_math_repair=True,
    math_thinking_budget_tokens=reasoning_math_budget,
    verifier_thinking_budget_tokens=reasoning_verifier_budget,
)

In [11]:
reasoning_preflight = reasoning_harness.preflight()
assert reasoning_preflight["reasoning"] == {"research": "none", "math": "enabled", "verifier": "enabled"}
reasoning_preflight

{'ready': True,
 'provider': 'bedrock',
 'model': 'us.anthropic.claude-sonnet-4-5-20250929-v1:0',
 'architecture': 'decomposed',
 'pipeline_variant': 'verification_with_repair',
 'reasoning': {'research': 'none', 'math': 'enabled', 'verifier': 'enabled'},
 'langfuse': 'authenticated',
 'qdrant_collection': 'sec_filing_chunks_v1'}

In [12]:
DataFrame([{
    "model": reasoning_harness.model_id, "research_thinking": 0,
    "math_thinking": reasoning_math_budget, "verifier_thinking": reasoning_verifier_budget,
}])

,model,research_thinking,math_thinking,verifier_thinking
0,us.anthropic.claude-sonnet-4-5-20250929-v1:0,0,4096,6000


## Complete reasoning-enabled batch

Run and save all 16 held-out questions before evaluation.

In [13]:
reasoning_runs = reasoning_harness.run_batch(ticker)
assert len(reasoning_runs) == len(gold_records)
reasoning_results_dir = save_run_records(reasoning_runs)

In [14]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status,
     "failure": run.failure_category}
    for run in reasoning_runs
])

,question_id,status,failure
0,ko-01,success,None
1,ko-02,success,None
2,ko-03,success,None
3,ko-04,success,None
4,ko-05,success,None
5,ko-06,success,None
6,ko-07,success,None
7,ko-08,success,None
8,ko-09,success,None
9,ko-10,success,None


## Inspect every reasoning-enabled answer

Review the full answers before aggregate scoring.

In [15]:
DataFrame([
    {"question_id": run.question.question_id, "status": run.status,
     "answer": run.answer.answer_text if run.answer else None}
    for run in reasoning_runs
])

,question_id,status,answer
0,ko-01,success,The Coca-Cola Company's consolidated net opera...
1,ko-02,success,The Coca-Cola Company's consolidated operating...
2,ko-03,success,North America generated the most operating inc...
3,ko-04,success,Coca-Cola's EMEA operating income in fiscal 20...
4,ko-05,success,The Coca-Cola Company's operating margin in fi...
5,ko-06,success,The Coca-Cola Company's net profit margin in f...
6,ko-07,success,Operating cash flow represented 15.45% of The ...
7,ko-08,success,The Coca-Cola Company's operating cash flow as...
8,ko-09,success,"Between fiscal 2024 and fiscal 2025, Coca-Cola..."
9,ko-10,success,"From fiscal 2024 to fiscal 2025, Coca-Cola's n..."


## Deterministic reasoning-enabled evaluation

Evaluate the saved batch without making model calls.

In [16]:
reasoning_evaluation = evaluate_saved_results(results_directory=reasoning_results_dir)
reasoning_summary = reasoning_evaluation["summary"]
reasoning_question_metrics = reasoning_evaluation["question_rows"]
reasoning_evaluation_df = DataFrame(reasoning_question_metrics)
reasoning_evaluation_df

,run_id,question_id,category,architecture,pipeline_variant,status,answer_text,numerical_correct,numerical_issues,unexpected_values,...,tool_f1,tool_exact_match,latency_ms,input_tokens,output_tokens,reasoning_tokens,retries,structured_output_repairs,cost_usd,failure_category
0,ko-01-20260901T145337917990Z,ko-01,direct_extraction,decomposed,verification_with_repair,success,The Coca-Cola Company's consolidated net opera...,True,[],[],...,1.0,True,26560.71,6293,1272,0,0,0,0.037959,None
1,ko-02-20260901T145404480730Z,ko-02,direct_extraction,decomposed,verification_with_repair,success,The Coca-Cola Company's consolidated operating...,True,[],[],...,1.0,True,25381.91,6306,1510,0,0,0,0.041568,None
2,ko-03-20260901T145429863176Z,ko-03,direct_extraction,decomposed,verification_with_repair,success,North America generated the most operating inc...,True,[],[],...,1.0,True,47524.91,31161,2587,0,0,0,0.132288,None
3,ko-04-20260901T145517391940Z,ko-04,direct_extraction,decomposed,verification_with_repair,success,Coca-Cola's EMEA operating income in fiscal 20...,True,[],[],...,1.0,True,29042.27,15955,1618,0,0,0,0.072135,None
4,ko-05-20260901T145546436604Z,ko-05,single_formula,decomposed,verification_with_repair,success,The Coca-Cola Company's operating margin in fi...,True,[],[],...,1.0,True,29399.20,8511,1846,0,0,0,0.053223,None
5,ko-06-20260901T145615836718Z,ko-06,single_formula,decomposed,verification_with_repair,success,The Coca-Cola Company's net profit margin in f...,True,[],[],...,1.0,True,27586.26,8558,1739,0,0,0,0.051759,None
6,ko-07-20260901T145643424389Z,ko-07,single_formula,decomposed,verification_with_repair,success,Operating cash flow represented 15.45% of The ...,True,[],[],...,1.0,True,27746.98,8596,1825,0,0,0,0.053163,None
7,ko-08-20260901T145711172375Z,ko-08,single_formula,decomposed,verification_with_repair,success,The Coca-Cola Company's operating cash flow as...,True,[],[],...,0.8,False,38265.31,26328,2416,0,0,0,0.115224,None
8,ko-09-20260901T145749441158Z,ko-09,period_comparison,decomposed,verification_with_repair,success,"Between fiscal 2024 and fiscal 2025, Coca-Cola...",True,[],[],...,1.0,True,49167.51,12290,3849,0,0,0,0.094605,None
9,ko-10-20260901T145838609450Z,ko-10,period_comparison,decomposed,verification_with_repair,success,"From fiscal 2024 to fiscal 2025, Coca-Cola's n...",False,[],[Difference in Growth Rates=14.43 percent],...,0.8,False,66798.46,27254,4488,0,0,1,0.149082,None


### Evaluation by question

Show strict answer, evidence, formula, formatting, and audit checks.

In [17]:
reasoning_evaluation_df[[
    "question_id", "category", "status",
    "numerical_correct", "evidence_complete",
    "formula_correct", "formatting_correct",
    "audit_trail_complete", "unexpected_values",
]] 

,question_id,category,status,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete,unexpected_values
0,ko-01,direct_extraction,success,True,True,None,True,True,[]
1,ko-02,direct_extraction,success,True,True,None,True,True,[]
2,ko-03,direct_extraction,success,True,True,None,True,False,[]
3,ko-04,direct_extraction,success,True,True,None,True,True,[]
4,ko-05,single_formula,success,True,True,True,True,True,[]
5,ko-06,single_formula,success,True,True,True,True,True,[]
6,ko-07,single_formula,success,True,True,True,True,True,[]
7,ko-08,single_formula,success,True,True,True,True,True,[]
8,ko-09,period_comparison,success,True,True,False,True,False,[]
9,ko-10,period_comparison,success,False,True,True,True,False,[Difference in Growth Rates=14.43 percent]


## Haiku 4.5 reasoning-enabled review

Apply the same exploratory judge and rubric used above.

In [18]:
reasoning_judged = evaluate_saved_results(
    results_directory=reasoning_results_dir, rerun_judge=True,
)
reasoning_judge_metrics = reasoning_judged["judge_rows"] or []
reasoning_summary = reasoning_judged["summary"]

In [19]:
DataFrame(reasoning_judge_metrics, columns=(
    "question_id", "llm_answer_correctness_grade", "workflow_correctness",
    "chronological_coherence", "workflow_efficiency", "auditability",
    "retries", "failure", "explanation",
))

,question_id,llm_answer_correctness_grade,workflow_correctness,chronological_coherence,workflow_efficiency,auditability,retries,failure,explanation
0,ko-01,5,5,5,5,5,0,None,The run correctly extracted The Coca-Cola Comp...
1,ko-02,5,5,5,5,5,0,None,The run correctly executed a direct extraction...
2,ko-03,5,4,5,4,4,0,None,Answer correctly identifies North America as t...
3,ko-04,5,5,5,5,5,0,None,The research correctly extracted Coca-Cola's E...
4,ko-05,5,5,5,5,5,0,None,The run correctly answers that Coca-Cola's FY2...
5,ko-06,5,5,5,5,5,0,None,The run correctly identifies Coca-Cola's FY202...
6,ko-07,5,5,5,5,5,0,None,The final answer of 15.45% is exactly correct ...
7,ko-08,5,5,5,5,5,0,None,The run correctly calculates operating cash fl...
8,ko-09,3,4,5,4,3,0,None,The answer correctly identifies the growth rat...
9,ko-10,3,4,5,5,3,0,None,The answer provides all required values (23.29...


In [20]:
save_evaluation_results(
    reasoning_results_dir, reasoning_question_metrics, reasoning_summary,
    reasoning_evaluation["retrieval_rows"], judge_rows=reasoning_judge_metrics,
)

PosixPath('/home/atanas-vitanov/Documents/final_project_financial_agent/results/2026-09-01T150513676267Z-decomposed-verification_with_repair-us-anthropic-claude-sonnet-4-5-20250929-v1-0-KO')

## Final metrics

Show the complete reasoning-enabled result.

In [21]:
print(format_evaluation_summary(reasoning_summary))

Runs: 16
run_success_rate: 100.00%
numerical_correctness_rate: 75.00%
evidence_validity_rate: 100.00%
evidence_completeness_rate: 100.00%
formula_correctness_rate: 58.33%
formatting_correctness_rate: 87.50%
unsupported_number_rate: 3.12%
audit_trail_completeness_rate: 56.25%
math_repair_invocation_rate: 0.00%
math_repair_success_rate: 0.00%
tool_precision: 93.75%
tool_recall: 100.00%
tool_f1: 96.77%
tool_exact_match_rate: 87.50%
tool_invocation_rate: 100.00%
tool_execution_success_rate: 100.00%
conditional_execution_rate: 95.83%
evidence_recall_at_5: 100.00%
irrelevant_retrieval_rate_at_5: 87.92%
judge_success_rate: 100.00%
judge_cost_coverage_rate: 100.00%
structured_output_repair_rate: 6.25%
Average latency: 43482.96 ms
Input tokens: 322232
Output tokens: 46545
Reasoning tokens: 0
Retries: 0
Structured-output repairs: 1
Infrastructure failures: 0
Estimated cost: $1.664871
Judge llm answer correctness grade: 4.19/5
Judge workflow correctness: 4.44/5
Judge chronological coherence: 5.00

## Verified, no-verifier, and reasoning comparison

Compare the three saved decomposed batches without rerunning them.

In [24]:
if "decomposed_summary" not in globals():
    saved_verified_evaluation = load_run(
        latest=True, architecture="decomposed", ticker=ticker,
        pipeline_variant="verification_with_repair",
        reasoning_mode="research=none, math=none, verifier=none",
    )
    decomposed_summary = saved_verified_evaluation["summary"]

if "no_verifier_summary" not in globals():
    saved_no_verifier_evaluation = load_run(
        latest=True, architecture="decomposed", ticker=ticker,
        pipeline_variant="no_verification",
    )
    no_verifier_summary = saved_no_verifier_evaluation["summary"]

if "reasoning_summary" not in globals():
    saved_reasoning_evaluation = load_run(
        latest=True, architecture="decomposed", ticker=ticker,
        pipeline_variant="verification_with_repair",
        reasoning_mode="research=none, math=enabled, verifier=enabled",
    )
    reasoning_summary = saved_reasoning_evaluation["summary"]

reasoning_comparison_metrics = (
    "run_success_rate", "numerical_correctness_rate",
    "evidence_validity_rate", "evidence_completeness_rate",
    "formula_correctness_rate", "formatting_correctness_rate",
    "audit_trail_completeness_rate", "tool_f1",
    "evidence_recall_at_5", "irrelevant_retrieval_rate_at_5",
    "average_latency_ms", "total_input_tokens", "total_output_tokens",
    "total_reasoning_tokens", "total_estimated_cost_usd",
    "average_estimated_cost_usd",
    "average_judge_llm_answer_correctness_grade",
    "average_judge_workflow_correctness",
    "average_judge_chronological_coherence",
    "average_judge_workflow_efficiency",
    "average_judge_auditability", "judge_total_estimated_cost_usd",
    "judge_average_estimated_cost_usd", "judge_total_model_invocations",
    "math_repair_invocation_rate", "math_repair_success_rate",
    "structured_output_repair_rate", "total_retries",
)

In [25]:
DataFrame([
    {"metric": metric,
     "with_verifier_no_thinking": decomposed_summary.get(metric),
     "without_verifier": no_verifier_summary.get(metric),
     "with_verifier_reasoning": reasoning_summary.get(metric)}
    for metric in reasoning_comparison_metrics
]).set_index("metric")

,with_verifier_no_thinking,without_verifier,with_verifier_reasoning
metric,,,
run_success_rate,1.0,1.0,1.0
numerical_correctness_rate,0.6875,0.6875,0.75
evidence_validity_rate,1.0,1.0,1.0
evidence_completeness_rate,1.0,1.0,1.0
formula_correctness_rate,0.666667,0.75,0.583333
formatting_correctness_rate,0.875,0.875,0.875
audit_trail_completeness_rate,0.6875,0.6875,0.5625
tool_f1,0.983607,0.967742,0.967742
evidence_recall_at_5,1.0,1.0,1.0


In [26]:
DataFrame([{
    "variant": variant, "answer_cost_per_question": summary["average_estimated_cost_usd"],
    "judge_cost_per_review": summary["judge_average_estimated_cost_usd"],
    "all_in_cost_per_question": round(float(summary["average_estimated_cost_usd"]) + float(summary["judge_average_estimated_cost_usd"]), 6),
} for variant, summary in (
    ("with_verifier_no_thinking", decomposed_summary),
    ("without_verifier", no_verifier_summary),
    ("with_verifier_reasoning", reasoning_summary),
)]).set_index("variant")

,answer_cost_per_question,judge_cost_per_review,all_in_cost_per_question
variant,,,
with_verifier_no_thinking,0.09911925,0.01482669375,0.113946
without_verifier,0.101395125,0.01445530625,0.115850
with_verifier_reasoning,0.1040544375,0.01424424375,0.118299


### Metrics by question type

Show which question types gained or lost from extended thinking.

In [27]:
reasoning_category_rows = [
    {"variant": variant, "question_type": category, **metrics}
    for variant, summary in (
        ("with_verifier_no_thinking", decomposed_summary),
        ("without_verifier", no_verifier_summary),
        ("with_verifier_reasoning", reasoning_summary),
    )
    for category, metrics in summary["by_category"].items()
]
DataFrame(reasoning_category_rows).set_index(["question_type", "variant"])

,,run_count,numerical_correctness_rate,evidence_completeness_rate,formula_correctness_rate,formatting_correctness_rate
question_type,variant,,,,,
direct_extraction,with_verifier_no_thinking,4,1.00,1.0,NaN,1.0
multi_step,with_verifier_no_thinking,4,0.25,1.0,0.25,0.5
period_comparison,with_verifier_no_thinking,4,0.50,1.0,0.75,1.0
single_formula,with_verifier_no_thinking,4,1.00,1.0,1.00,1.0
direct_extraction,without_verifier,4,1.00,1.0,NaN,1.0
multi_step,without_verifier,4,0.25,1.0,0.25,0.5
period_comparison,without_verifier,4,0.50,1.0,1.00,1.0
single_formula,without_verifier,4,1.00,1.0,1.00,1.0
direct_extraction,with_verifier_reasoning,4,1.00,1.0,NaN,1.0


## Reasoning-enabled questions needing review

Questions not listed passed all selected checks.

In [28]:
reasoning_review = [
    row for row in reasoning_question_metrics
    if row["status"] != "success" or any(row[name] is not True for name in CHECKS)
]
DataFrame(reasoning_review, columns=(
    "question_id", "category", "status", "numerical_correct",
    "evidence_complete", "formula_correct", "formatting_correct",
    "audit_trail_complete",
))

,question_id,category,status,numerical_correct,evidence_complete,formula_correct,formatting_correct,audit_trail_complete
0,ko-03,direct_extraction,success,True,True,None,True,False
1,ko-09,period_comparison,success,True,True,False,True,False
2,ko-10,period_comparison,success,False,True,True,True,False
3,ko-12,period_comparison,success,True,True,False,True,False
4,ko-13,multi_step,success,False,True,True,True,True
5,ko-14,multi_step,success,False,True,False,False,False
6,ko-15,multi_step,success,False,True,False,False,False
7,ko-16,multi_step,success,True,True,False,True,False


## Failures

An empty table means all pipeline and judge calls completed.

In [29]:
reasoning_failures = [
    {"source": "pipeline", "question_id": run.question.question_id,
     "failure_category": run.failure_category, "failure_message": run.failure_message}
    for run in reasoning_runs if run.status == "failure"
] + [
    {"source": "judge", "question_id": row["question_id"],
     "failure_category": "judge_failure", "failure_message": row["failure"]}
    for row in reasoning_judge_metrics if row["failure"] is not None
]
reasoning_failures_df = DataFrame(reasoning_failures, columns=(
    "source", "question_id", "failure_category", "failure_message",
))
reasoning_failures_df

,source,question_id,failure_category,failure_message


## Saved reasoning artifacts

Save the evaluation and failure tables beside the run records.

In [30]:
reasoning_evaluation_df.to_csv(reasoning_results_dir / "evaluations.csv", index=False)
reasoning_failures_df.to_csv(reasoning_results_dir / "failures.csv", index=False)
DataFrame([{
    "artifact": name,
    "path": str(reasoning_results_dir / name),
} for name in ("runs.jsonl", "summary.json", "judge_records.jsonl",
              "evaluations.csv", "failures.csv")])

,artifact,path
0,runs.jsonl,/home/atanas-vitanov/Documents/final_project_f...
1,summary.json,/home/atanas-vitanov/Documents/final_project_f...
2,judge_records.jsonl,/home/atanas-vitanov/Documents/final_project_f...
3,evaluations.csv,/home/atanas-vitanov/Documents/final_project_f...
4,failures.csv,/home/atanas-vitanov/Documents/final_project_f...


## Reasoning ablation findings and conclusion

Extended thinking raised strict numerical correctness from **68.75% to 75.00%**, but reduced formula correctness from **66.67% to 58.33%** and audit completeness from **68.75% to 56.25%**. Evidence remained complete. Average latency increased from 29.14 to 43.48 seconds, and answer cost rose from **\$0.0991** to **\$0.1041** per question. Haiku's answer grade fell from 4.56/5 to 4.19/5.

The only strict numerical gain was **KO-11**, where the reasoning run omitted two unnecessary annual asset values. **KO-13** gained a valid formula and complete audit path. These gains were offset by regressions: **KO-09** omitted the 35.86-point comparison, **KO-12** omitted the 26.45-point comparison, and **KO-15** used a non-canonical percentage path.

Every reasoning review row explained:

- **KO-03:** the answer is correct; the same alternative-source mapping causes the audit failure.
- **KO-09:** both rates are correct, but the required 35.86-point difference and calculation are missing.
- **KO-10:** the answer is correct; an extra 14.43-point structured value and period metadata fail strict checks.
- **KO-12:** both rates are correct, but the required 26.45-point difference is not stated or calculated.
- **KO-13:** the content and calculation path are correct; required margin changes are still marked as extra structured values.
- **KO-14:** the answer rounds 45.60%, 36.84%, and −8.76 points too coarsely.
- **KO-15:** the required values are correct, but an extra FY2025 value, alternate formula, and negative-value formatting fail strict checks.
- **KO-16:** the four rates are correct, but the slowest measure and 35.86-point spread are not stated or calculated.

### No-thinking versus reasoning comparison

| Metric | No thinking | Reasoning |
|---|---:|---:|
| Successful runs | 16/16 | 16/16 |
| Numerical correctness | 68.75% | **75.00%** |
| Formula correctness | **66.67%** | 58.33% |
| Audit-trail completeness | **68.75%** | 56.25% |
| Evidence completeness | 100.00% | 100.00% |
| Tool F1 | **98.36%** | 96.77% |
| Average answer latency | **29.14 s** | 43.48 s |
| Total answer tokens | 399,640 | **368,777** |
| Separately recorded reasoning tokens | 0 | 0 |
| Average answer cost | **\$0.0991** | \$0.1041 |
| Average judge cost | \$0.0148 | **\$0.0142** |
| Average answer + judge cost | **\$0.1139** | \$0.1183 |
| Haiku answer grade | **4.56/5** | 4.19/5 |
| Haiku workflow grade | **4.50/5** | 4.44/5 |
| Haiku auditability grade | 4.38/5 | 4.38/5 |

The saved usage records do not expose a separate reasoning-token count. The zero shown above does not mean thinking was disabled; it means this notebook cannot measure those tokens separately.

### Average answer price by question type

This is the average cost to generate one answer. Haiku evaluation cost is excluded.

| Question type | Monolithic | Verified, no thinking | No Verifier | Verified + reasoning |
|---|---:|---:|---:|---:|
| Direct extraction | **\$0.0422** | \$0.0944 | \$0.0672 | \$0.0710 |
| Single formula | **\$0.0442** | \$0.0457 | \$0.0648 | \$0.0683 |
| Period comparison | **\$0.1036** | \$0.1543 | \$0.1650 | \$0.1635 |
| Multi-step | **\$0.0784** | \$0.1021 | \$0.1086 | \$0.1134 |

Period comparisons were the most expensive question type for every architecture. The monolithic agent was cheapest in every category.

### Final takeaway

For this project, the **decomposed approach with a verifier** is the best overall choice. It gives a clear trail showing where each number came from and how it was calculated, and it was much more accurate than the single-agent approach. In these runs, however, the verifier did not change any final answer. That means it is valuable as a safety check and audit record, but this benchmark does not prove that it fixed mistakes.

Letting the decomposed system spend more time thinking produced the highest numerical score: **75.00%**, compared with **68.75%** without extra thinking and **37.50%** for the single agent. But it was slower, cost more, and received slightly weaker scores from the LLM reviewer. In plain terms, extra thinking helped some answers, but did not make every part of the response better.

### A practical way to use this in production

Rather than use one approach for every question, a production system could choose the simplest reliable path for each kind of request:

| Question class | Proposed route | Reason |
|---|---|---|
| Direct extraction | Monolithic agent | It was quick, inexpensive, and correct on all four direct-look-up questions. |
| Simple single-formula | Monolithic agent | Use the faster route when the formula and the input values are clear; otherwise send it to the verified route. |
| Period comparison | Decomposed pipeline with verification | Comparing years is easier to check when the evidence and calculations are shown step by step. |
| Hard or multi-step | Decomposed pipeline with extended thinking and verification | Use the more careful route when several calculations or decisions depend on one another. |

This is a sensible starting point, not a final rule. The single agent was only correct on two of the four single-formula questions, and extra thinking did not improve the multi-step questions in this small test.